In [12]:
import duckdb
import pandas as pd
from pathlib import Path

print("🔄 Воспроизведение Airflow DAG")

# Подключаемся к DuckDB
con = duckdb.connect()

# Правильные пути (поднимаемся на уровень вверх через ..)
ratings_path = Path('..', 'data', 'raw', 'ml-25m', 'ratings.csv')
movies_path = Path('..', 'data', 'raw', 'ml-25m', 'movies.csv')

print(f"\n📥 Загрузка данных...")
print(f"   ratings: {ratings_path}")
print(f"   movies: {movies_path}")

# Проверяем что файлы существуют
if not ratings_path.exists():
    raise FileNotFoundError(f"Файл не найден: {ratings_path.absolute()}")
if not movies_path.exists():
    raise FileNotFoundError(f"Файл не найден: {movies_path.absolute()}")

# Загружаем через pandas
ratings_df = pd.read_csv(ratings_path)
movies_df = pd.read_csv(movies_path)

print(f"raw_ratings: {len(ratings_df):,} строк")
print(f"raw_movies: {len(movies_df):,} строк")

# Регистрируем в DuckDB
con.register('ratings_temp', ratings_df)
con.register('movies_temp', movies_df)

# Создаем таблицы
con.execute("CREATE TABLE raw_ratings AS SELECT * FROM ratings_temp")
con.execute("CREATE TABLE raw_movies AS SELECT * FROM movies_temp")

# Трансформация (как в Airflow)
print("\n🔄 Трансформация данных")
con.execute("""
    CREATE TABLE user_activity AS
    SELECT 
        r.userId, r.movieId, r.rating, r.timestamp,
        to_timestamp(r.timestamp) AS datetime,
        CAST(to_timestamp(r.timestamp) AS DATE) AS date,
        DATE_PART('hour', to_timestamp(r.timestamp)) AS hour,
        m.title, m.genres
    FROM raw_ratings r
    LEFT JOIN raw_movies m ON r.movieId = m.movieId
""")

user_count = con.execute("SELECT COUNT(*) FROM user_activity").fetchone()[0]
print(f"user_activity: {user_count:,} строк")

# Метрики
print("\n📊 Расчет метрик")
metrics = con.execute("""
    SELECT 
        COUNT(DISTINCT userId) as dau,
        AVG(rating) as avg_rating,
        COUNT(*) as total_ratings
    FROM user_activity
    WHERE date = (SELECT MAX(date) FROM user_activity)
""").fetchall()

print(f"DAU: {metrics[0][0]:,}")
print(f"Avg Rating: {metrics[0][1]:.2f}")
print(f"Total Ratings: {metrics[0][2]:,}")

🔄 Воспроизведение Airflow DAG

📥 Загрузка данных...
   ratings: ..\data\raw\ml-25m\ratings.csv
   movies: ..\data\raw\ml-25m\movies.csv
raw_ratings: 25,000,095 строк
raw_movies: 62,423 строк

🔄 Трансформация данных
user_activity: 25,000,095 строк

📊 Расчет метрик
DAU: 114
Avg Rating: 3.70
Total Ratings: 502
